## Stereo Image Mosaic - TRY OUT

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

def load_tiff(image_path):
    """Specialized TIFF loader with fallback options"""
    try:
        # Try OpenCV first
        img = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if img is not None:
            return img
        
        # Fallback to PIL if OpenCV fails
        from PIL import Image
        img = np.array(Image.open(image_path))
        if len(img.shape) == 2:  # Grayscale
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        else:  # RGB
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        return img
    except Exception as e:
        print(f"Error loading {image_path}: {str(e)}")
        return None

def enhanced_mosaic(img1_path, img2_path, output_path='mosaic_result.tif'):
    # Load images
    print("Loading images...")
    img1 = load_tiff(img1_path)
    img2 = load_tiff(img2_path)
    
    if img1 is None or img2 is None:
        print("Failed to load one or both images")
        return

    # Resize if too large (adjust max_dim as needed)
    max_dim = 4000
    h, w = img1.shape[:2]
    if max(h, w) > max_dim:
        scale = max_dim / max(h, w)
        img1 = cv2.resize(img1, None, fx=scale, fy=scale)
        img2 = cv2.resize(img2, None, fx=scale, fy=scale)
        print(f"Resized images to {img1.shape[1]}x{img1.shape[0]}")

    # Convert to grayscale for feature detection
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    # Feature detection with SIFT
    print("Detecting features...")
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(gray1, None)
    kp2, des2 = sift.detectAndCompute(gray2, None)

    # Feature matching with FLANN
    print("Matching features...")
    FLANN_INDEX_KDTREE = 1
    index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)
    
    matches = flann.knnMatch(des1, des2, k=2)
    
    # Lowe's ratio test
    good_matches = []
    for m, n in matches:
        if m.distance < 0.7 * n.distance:
            good_matches.append(m)
    
    if len(good_matches) < 10:
        print(f"Only {len(good_matches)} good matches found - mosaic may fail")
    
    # Calculate homography
    print("Calculating transformation...")
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1,1,2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1,1,2)
    
    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
    
    # Warp second image
    print("Warping images...")
    h, w = img1.shape[:2]
    img2_warped = cv2.warpPerspective(img2, H, (w*2, h))
    
    # Simple blending (replace with multi-band for better results)
    print("Blending images...")
    result = img2_warped.copy()
    result[0:h, 0:w] = img1
    
    # Save result
    cv2.imwrite(output_path, result)
    print(f"Mosaic saved to {output_path}")
    
    # Display
    plt.figure(figsize=(20,10))
    plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

# Run on your specific files
img1_path = r'W:\1945_02_17#_#US7_0092_A\8082#_#US7_0092_A#ID0170#17_02_1945.tif'
img2_path = r'W:\1945_02_17#_#US7_0092_A\8083#_#US7_0092_A#ID0170#17_02_1945.tif'
enhanced_mosaic(img1_path, img2_path)

## Image Border Cleanup

### Environment and Path check

In [ ]:
def environment_summary():
    import sys
    import os
    from platform import python_version
    
    print("=== ENVIRONMENT SUMMARY ===")
    print(f"Python version: {python_version()}")
    print(f"Python executable: {sys.executable}")
    print(f"Current working directory: {os.getcwd()}")
    print(f"Platform: {sys.platform}")
    print(f"Prefix: {sys.prefix}")
    
    # Check virtual environment
    if hasattr(sys, 'base_prefix') and sys.prefix != sys.base_prefix:
        print("Virtual environment: Yes")
    else:
        print("Virtual environment: No")
    
    # Check conda environment
import numpy as np
from PIL import Image
import rasterio
from rasterio.errors import RasterioIOError
from samgeo import SamGeo

# ---------------------------------------------------------
# USER SETTINGS
# ---------------------------------------------------------
INPUT = r"/mnt/data/e874dc29-6476-46c2-a6df-39030e57bd2b.png"
OUTPUT = r"/mnt/data/output_cleaned.png"

MAX_SAM_SIZE = 1024
MIN_SAM_SIZE = 512

MODEL_TYPE = "vit_b"
CHECKPOINT = "sam_vit_b_01ec64.pth"
DEVICE = "cpu"

# ---------------------------------------------------------
# UNIVERSAL LOAD FUNCTION
# ---------------------------------------------------------
def load_any(path):
    try:
        with rasterio.open(path) as src:
            arr = src.read()

        if arr.shape[0] >= 3:
            rgb = np.moveaxis(arr[:3], 0, -1)
        else:
            band = arr[0]
            rgb = np.stack([band, band, band], axis=-1)

        print("Loaded with rasterio")
        return rgb.astype(np.uint8)

    except:
        img = Image.open(path).convert("RGB")
        print("Loaded with Pillow")
        return np.array(img, dtype=np.uint8)

# ---------------------------------------------------------
# LOAD IMAGE
# ---------------------------------------------------------
orig = load_any(INPUT)
H, W = orig.shape[:2]

# ---------------------------------------------------------
# DOWNSCALE For SAM (with MIN SIZE PROTECTION)
# ---------------------------------------------------------
scale = MAX_SAM_SIZE / max(W, H)
newW = max(int(W * scale), MIN_SAM_SIZE)
newH = max(int(H * scale), MIN_SAM_SIZE)

sam_input = np.array(
    Image.fromarray(orig).resize((newW, newH), Image.BILINEAR),
    dtype=np.uint8
)

print("SAM working size:", newW, "x", newH)

# ---------------------------------------------------------
# RUN SAM
# ---------------------------------------------------------
sam = SamGeo(model_type=MODEL_TYPE, checkpoint=CHECKPOINT, device=DEVICE)

print("Generating masks...")

masks = sam.generate(sam_input)

# If SAM returned None or empty list → abort safely
if masks is None or len(masks) == 0:
    raise RuntimeError("ERROR: SAM did not return any masks. "
                       "Try increasing MAX_SAM_SIZE or ensure the input is valid RGB.")

print("Masks returned:", len(masks))

# ---------------------------------------------------------
# FIND LARGEST MASK
# ---------------------------------------------------------
largest = max(masks, key=lambda m: m["area"])
mask_small = largest["segmentation"].astype(np.uint8) * 255

# ---------------------------------------------------------
# UPSCALE MASK TO ORIGINAL SIZE
# ---------------------------------------------------------
mask_up = np.array(
    Image.fromarray(mask_small).resize((W, H), Image.NEAREST)
)

# ---------------------------------------------------------
# EXPORT
# ---------------------------------------------------------
rgba = np.dstack((orig, mask_up))
Image.fromarray(rgba).save(OUTPUT)

print("Saved:", OUTPUT)
    conda_env = os.environ.get('CONDA_DEFAULT_ENV', 'Not in conda environment')
    print(f"Conda environment: {conda_env}")

environment_summary()

### Main Code

In [7]:
import numpy as np
import cv2
from PIL import Image
import os

INPUT = r"G:\0 BIPLOV\TASKS\Jonathan\IMageBorderCleanup\TRY\1018#16_1411#ID1472#27_11_1944_modified.tif"
OUTDIR = os.path.dirname(INPUT)

# ---------------------------------------------------------
# Load as RGB + grayscale
# ---------------------------------------------------------
img = np.array(Image.open(INPUT).convert("RGB"))
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
h, w = gray.shape


# ---------------------------------------------------------
# Utility: Get largest connected component from a binary mask
# ---------------------------------------------------------
def largest_cc(bin_mask):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bin_mask)
    if num_labels <= 1:
        return bin_mask
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return (labels == largest).astype(np.uint8) * 255


# ---------------------------------------------------------
# Utility: Auto-crop image using mask bounding box
# ---------------------------------------------------------
def autocrop(image, mask):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return image  # no cropping possible
    xmin, xmax = xs.min(), xs.max()
    ymin, ymax = ys.min(), ys.max()
    return image[ymin:ymax, xmin:xmax]


# ---------------------------------------------------------
# METHOD 1 — Largest Connected Component (from high-pass)
# ---------------------------------------------------------
_, th_manual = cv2.threshold(gray, 0, 255, cv2.THRESH_OTSU)
lcc_mask = largest_cc((gray < th_manual).astype(np.uint8) * 255)
lcc_crop = autocrop(img, lcc_mask)
Image.fromarray(lcc_crop).save(os.path.join(OUTDIR, "result_largest_cc.tif"))


# ---------------------------------------------------------
# METHOD 2 — OTSU Threshold
# ---------------------------------------------------------
_, otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
otsu_mask = largest_cc(otsu)
otsu_crop = autocrop(img, otsu_mask)
Image.fromarray(otsu_crop).save(os.path.join(OUTDIR, "result_otsu.tif"))


# ---------------------------------------------------------
# METHOD 3 — Adaptive Threshold
# ---------------------------------------------------------
adaptive = cv2.adaptiveThreshold(
    gray, 255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY_INV,
    51, 10
)
adaptive_mask = largest_cc(adaptive)
adaptive_crop = autocrop(img, adaptive_mask)
Image.fromarray(adaptive_crop).save(os.path.join(OUTDIR, "result_adaptive.tif"))


# ---------------------------------------------------------
# METHOD 4 — Morphological Cleanup (Refined Borders)
# ---------------------------------------------------------
blur = cv2.GaussianBlur(gray, (7, 7), 0)
_, basic = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# open → remove noise, close → seal gaps
clean = cv2.morphologyEx(basic, cv2.MORPH_OPEN, np.ones((5,5), np.uint8))
clean = cv2.morphologyEx(clean, cv2.MORPH_CLOSE, np.ones((15,15), np.uint8))

clean_mask = largest_cc(clean)
clean_crop = autocrop(img, clean_mask)
Image.fromarray(clean_crop).save(os.path.join(OUTDIR, "result_morphological.tif"))


print("✔ 4 mask-based results saved:")
print("- result_largest_cc.tif")
print("- result_otsu.tif")
print("- result_adaptive.tif")
print("- result_morphological.tif")


✔ 4 mask-based results saved:
- result_largest_cc.tif
- result_otsu.tif
- result_adaptive.tif
- result_morphological.tif
